# Taal Vista Hotel Web Scraping

## Purpose

This notebook collects publicly available information from permitted pages of the official Taal Vista Hotel website.

The workflow begins by checking website access rules and inspecting page structure. Data extraction will proceed gradually, beginning with accommodation information.

## Data Collection Principles

1. Collect only publicly accessible business information.

2. Respect website access rules and technical restrictions.

3. Use reasonable request delays.

4. Do not access the external booking system.

5. Do not collect unnecessary personal information.

6. Preserve source URLs and collection dates.

7. Save extracted information in `data/raw` without analytical cleaning.

In [1]:
from pathlib import Path
from datetime import date
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
BASE_URL = "https://www.taalvistahotel.com"
ROBOTS_URL = f"{BASE_URL}/robots.txt"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

robots_response = requests.get(
    ROBOTS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", robots_response.status_code)
print()
print(robots_response.text[:3000])

Status code: 200

User-agent: *
Crawl-Delay: 3



## Website Access Check

The website returned asuccessful response for `robots.txt`.

The published rules apply to all automated user agents and specify a crawl delay of three seconds. No disallowed paths were listed.

This project will therefore wait at least three seconds between page requests and will stop if the website returns an access restriction or rate limit response.

In [3]:
robot_parser = RobotFileParser()
robot_parser.set_url(ROBOTS_URL)
robot_parser.parse(robots_response.text.splitlines())

ROOMS_URL = f"{BASE_URL}/rooms/"

print("Rooms page permitted:", robot_parser.can_fetch(HEADERS["User-Agent"], ROOMS_URL))
print("Required crawl delay:", robot_parser.crawl_delay(HEADERS["User-Agent"]))

Rooms page permitted: True
Required crawl delay: 3


In [6]:
import time
import requests

BASE_URL = "https://www.taalvistahotel.com"
ROOMS_URL = f"{BASE_URL}/rooms/"

HEADERS = {
    "User-Agent": "TaalVistaHotelResearchProject/1.0"
}

time.sleep(3)

rooms_response = requests.get(
    ROOMS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", rooms_response.status_code)
print("Final URL:", rooms_response.url)
print("Content type:", rooms_response.headers.get("Content-Type"))
print("HTML characters:", len(rooms_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/rooms/
Content type: text/html; charset=UTF-8
HTML characters: 999862


## Rooms Page Access Result

The rooms page returned a successful HTTP status code of 200 and HTML content.

The page will first be inspected before extraction. This helps identify the correct HTML elements and prevents the scraper from collecting navigation, footer, script, and unrelated page content.

In [7]:
rooms_soup = BeautifulSoup(
    rooms_response.text,
    "lxml"
)

page_title = rooms_soup.title.get_text(
    strip=True
) if rooms_soup.title else "No title found"

print("Page title:", page_title)

Page title: ROOMS - Taal Vista Hotel


In [8]:
page_headings = []

for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text and heading_text not in page_headings:
        page_headings.append(heading_text)

print("Unique headings found:", len(page_headings))
print()

for heading_text in page_headings[:50]:
    print(heading_text)

Unique headings found: 15

Rooms
DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


## Initial Rooms Page Findings

The heading inspection identified possible room and suite names together with unrelated website content.

The extraction will not rely only on heading text. The surrounding HTML structure must be inspected to determine whether each room has a description, wing, bed type, view, capacity, room size, or amenities.

In [9]:
excluded_headings = [
    "Rooms",
    "Privacy Overview",
    "Chinese New Year Offerings at Taal Vista Hotel",
    "Breathe Fresh"
]

room_headings = []

for heading_text in page_headings:
    if heading_text not in excluded_headings:
        room_headings.append(heading_text)

print("Possible room headings:", len(room_headings))
print()

for room_heading in room_headings:
    print(room_heading)

Possible room headings: 11

DELUXE ROOM
PREMIER QUEEN ROOM
Two-Bedroom Deluxe Suite
Batangas Suite
superior room
deluxe room
ridge room
PREMIER ROOM
ONE-BEDROOM DELUXE SUITE
TAAL SUITE
TAGAYTAY SUITE


In [10]:
for heading in rooms_soup.find_all(["h1", "h2", "h3", "h4"]):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text in room_headings:
        print("Room:", heading_text)
        print("Heading tag:", heading.name)
        print("Heading class:", heading.get("class"))
        print("Parent tag:", heading.parent.name)
        print("Parent class:", heading.parent.get("class"))
        print("=" * 50)

Room: DELUXE ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Room: PREMIER QUEEN ROOM
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-5629', 'style-local-115-c35']
Room: Two-Bedroom Deluxe Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c44']
Room: Batangas Suite
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c49']
Room: superior room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c60']
Room: deluxe room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c64']
Room: ridge room
Heading tag: h3
Heading class: []
Parent tag: div
Parent class: ['h-heading__outer', 'style-1580', 'style-local-115-c7

In [11]:
sample_room_names = [
    "DELUXE ROOM",
    "PREMIER QUEEN ROOM"
]

for sample_room_name in sample_room_names:
    sample_heading = rooms_soup.find(
        "h3",
        string=lambda text: text and text.strip() == sample_room_name
    )

    print("ROOM:", sample_room_name)

    for level, ancestor in enumerate(sample_heading.parents):
        if level >= 7:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:400])

    print()
    print("=" * 70)

ROOM: DELUXE ROOM

Ancestor level: 0
Tag: div
Class: ['h-heading__outer', 'style-1580', 'style-local-115-c30']
Text preview: DELUXE ROOM

Ancestor level: 1
Tag: div
Class: ['h-global-transition-all', 'h-heading', 'style-1580', 'style-local-115-c30', 'position-relative', 'h-element']
Text preview: DELUXE ROOM

Ancestor level: 2
Tag: div
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-100', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: DELUXE ROOM Stay in our newly renovated Deluxe Rooms, thoughtfully designed for quiet and intimate accommodation. Located in the Lake Wing, guests may choose between a King, Twin, or Queen bed to suit their preference. Each room
                                      offers views of Aguinaldo Highway or the hotel’s charming courtyard, with easy access to our exclusive gate to Skyranch for added lei

Ancestor level: 3
Tag: div
Class: ['d-flex', 'h-flex-basis', 'h-column__inner', 'h-px-

## Initial Room Record Extraction

The repeated `h-column__content` container holds each room heading and its associated descriptive content.

The first extraction will preserve the room names and descriptions as published. Capitalization, category standardization, and feature extraction will be handled later during data cleaning.

In [12]:
room_records = []

for heading in rooms_soup.find_all("h3"):
    room_name = heading.get_text(" ", strip=True)

    if room_name in room_headings:
        room_container = heading.find_parent(
            "div",
            class_="h-column__content"
        )

        paragraph_texts = []

        if room_container:
            for paragraph in room_container.find_all("p"):
                paragraph_text = paragraph.get_text(" ", strip=True)

                if paragraph_text:
                    paragraph_texts.append(paragraph_text)

        room_description = " ".join(paragraph_texts)

        room_records.append({
            "room_name": room_name,
            "room_description": room_description,
            "source_url": ROOMS_URL,
            "date_collected": date.today().isoformat()
        })

rooms_raw_df = pd.DataFrame(room_records)

print("Rows:", rooms_raw_df.shape[0])
print("Columns:", rooms_raw_df.shape[1])
print()
print("Missing values:")
print(rooms_raw_df.isnull().sum())

display(
    rooms_raw_df[
        ["room_name", "room_description"]
    ]
)

Rows: 11
Columns: 4

Missing values:
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64


,room_name,room_description
0,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou..."
1,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...
2,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ..."
3,Batangas Suite,Celebrate life’s milestones or simply unwind i...
4,superior room,Surrounded by the refreshing view of lush gree...
5,deluxe room,"Located on a higher floor, the rooms are taste..."
6,ridge room,Inspired by the rich culture and relaxing outd...
7,PREMIER ROOM,"For a more captivating and serene stay, our Pr..."
8,ONE-BEDROOM DELUXE SUITE,Features a spacious living room and the privac...
9,TAAL SUITE,Features a modern and fresh vibe with floor-to...


## Raw Room Data Quality Check

The initial extraction returned 11 room records with no missing descriptions.

Room names will be checked without considering capitalization. A repeated normalized name will not automatically be treated as a duplicate because the records may represent distinct products in different hotel wings.

In [13]:
rooms_raw_df.insert(
    0,
    "room_record_id",
    range(1, len(rooms_raw_df) + 1)
)

rooms_raw_df["description_length"] = (
    rooms_raw_df["room_description"].str.len()
)

rooms_raw_df["normalized_room_name"] = (
    rooms_raw_df["room_name"]
    .str.strip()
    .str.lower()
)

duplicate_name_check = rooms_raw_df[
    rooms_raw_df.duplicated(
        subset="normalized_room_name",
        keep=False
    )
]

print("Exact duplicate rows:", rooms_raw_df.duplicated().sum())
print(
    "Repeated normalized room names:",
    duplicate_name_check.shape[0]
)
print()

display(
    duplicate_name_check[
        [
            "room_record_id",
            "room_name",
            "description_length",
            "room_description"
        ]
    ]
)

Exact duplicate rows: 0
Repeated normalized room names: 2



,room_record_id,room_name,description_length,room_description
0,1,DELUXE ROOM,789,"Stay in our newly renovated Deluxe Rooms, thou..."
5,6,deluxe room,336,"Located on a higher floor, the rooms are taste..."


## Raw Data Export Decision

The two Deluxe Room records will be preserved because they have different descriptions and may represent distinct room products.

Helper columns created for quality checking will not be included in the raw export. The raw dataset will contain only the record identifier, original room name, original description, source URL, and collection date.

In [14]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_data_folder = project_root / "data" / "raw"
raw_data_folder.mkdir(parents=True, exist_ok=True)

rooms_export_df = rooms_raw_df[
    [
        "room_record_id",
        "room_name",
        "room_description",
        "source_url",
        "date_collected"
    ]
].copy()

rooms_file = raw_data_folder / "taal_vista_rooms_raw.csv"

rooms_export_df.to_csv(
    rooms_file,
    index=False
)

print("File saved:", rooms_file)
print("Rows saved:", rooms_export_df.shape[0])
print("Columns saved:", rooms_export_df.shape[1])
print("File exists:", rooms_file.exists())

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_rooms_raw.csv
Rows saved: 11
Columns saved: 5
File exists: True


## Raw CSV Verification

The exported CSV will be loaded again to confirm that it is readable and that its dimensions, columns, missing values, and duplicate records remain correct after export.

In [15]:
rooms_verification_df = pd.read_csv(rooms_file)

print("Loaded shape:", rooms_verification_df.shape)
print()
print("Columns:")
print(rooms_verification_df.columns.tolist())
print()
print("Missing values:")
print(rooms_verification_df.isnull().sum())
print()
print(
    "Exact duplicate records:",
    rooms_verification_df.duplicated().sum()
)

display(rooms_verification_df.head())

Loaded shape: (11, 5)

Columns:
['room_record_id', 'room_name', 'room_description', 'source_url', 'date_collected']

Missing values:
room_record_id      0
room_name           0
room_description    0
source_url          0
date_collected      0
dtype: int64

Exact duplicate records: 0


,room_record_id,room_name,room_description,source_url,date_collected
0,1,DELUXE ROOM,"Stay in our newly renovated Deluxe Rooms, thou...",https://www.taalvistahotel.com/rooms/,2026-09-02
1,2,PREMIER QUEEN ROOM,Unwind in the newly renovated Premier Queen Ro...,https://www.taalvistahotel.com/rooms/,2026-09-02
2,3,Two-Bedroom Deluxe Suite,"Ideal for families or small groups, the newly ...",https://www.taalvistahotel.com/rooms/,2026-09-02
3,4,Batangas Suite,Celebrate life’s milestones or simply unwind i...,https://www.taalvistahotel.com/rooms/,2026-09-02
4,5,superior room,Surrounded by the refreshing view of lush gree...,https://www.taalvistahotel.com/rooms/,2026-09-02


# Dining Data Collection

## Purpose

This section checks the official dining page and identifies publicly presented restaurants, bars, cafés, operating schedules, cuisines, and descriptions.

The page structure will be inspected before any dining records are extracted.

In [16]:
DINING_URL = f"{BASE_URL}/dining/"

print(
    "Dining page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        DINING_URL
    )
)

Dining page allowed: True


In [17]:
time.sleep(3)

dining_response = requests.get(
    DINING_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", dining_response.status_code)
print("Final URL:", dining_response.url)
print("Content type:", dining_response.headers.get("Content-Type"))
print("HTML characters:", len(dining_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/dining/
Content type: text/html; charset=UTF-8
HTML characters: 998709


## Dining Page Structure Inspection

The dining page returned a successful response. The page headings will be inspected to identify possible restaurants, bars, cafés, and unrelated website content.

In [18]:
dining_soup = BeautifulSoup(
    dining_response.text,
    "lxml"
)

dining_page_title = (
    dining_soup.title.get_text(strip=True)
    if dining_soup.title
    else "No title found"
)

print("Page title:", dining_page_title)

Page title: DINING - Taal Vista Hotel


In [19]:
dining_headings = []

for heading in dining_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text and heading_text not in dining_headings:
        dining_headings.append(heading_text)

print(
    "Unique headings found:",
    len(dining_headings)
)
print()

for heading_text in dining_headings[:50]:
    print(heading_text)

Unique headings found: 12

Dining
Broad and Open
VERANDA
Sustainable and Natural
TĀZA FRESH TABLE
Grand and Iconic
LOBBY LOUNGE
pasalubong
alta ridge bar
Privacy Overview
Chinese New Year Offerings at Taal Vista Hotel
Breathe Fresh


In [20]:
for heading in dining_soup.find_all(
    ["h1", "h2", "h3", "h4"]
):
    heading_text = heading.get_text(" ", strip=True)

    if heading_text:
        print("Heading:", heading_text)
        print("Tag:", heading.name)
        print("Class:", heading.get("class"))
        print("Parent class:", heading.parent.get("class"))
        print("=" * 50)

Heading: Dining
Tag: h3
Class: []
Parent class: ['h-heading__outer', 'style-5812', 'style-local-116-c8']
Heading: Broad and Open
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c32']
Heading: VERANDA
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c33']
Heading: Sustainable and Natural
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c46']
Heading: TĀZA FRESH TABLE
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c47']
Heading: Grand and Iconic
Tag: h4
Class: []
Parent class: ['h-heading__outer', 'style-380', 'style-local-116-c77']
Heading: LOBBY LOUNGE
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c78']
Heading: pasalubong
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-116-c90']
Heading: alta ridge bar
Tag: h2
Class: []
Parent class: ['h-heading__outer', 'style-401', 'style-local-1

In [21]:
sample_outlet_names = [
    "VERANDA",
    "pasalubong"
]

for sample_outlet_name in sample_outlet_names:
    sample_heading = dining_soup.find(
        "h2",
        string=lambda text: (
            text
            and text.strip() == sample_outlet_name
        )
    )

    print("OUTLET:", sample_outlet_name)

    for level, ancestor in enumerate(
        sample_heading.parents
    ):
        if level >= 7:
            break

        ancestor_text = ancestor.get_text(
            " ",
            strip=True
        )

        print()
        print("Ancestor level:", level)
        print("Tag:", ancestor.name)
        print("Class:", ancestor.get("class"))
        print("Text preview:", ancestor_text[:500])

    print()
    print("=" * 70)

OUTLET: VERANDA

Ancestor level: 0
Tag: div
Class: ['h-heading__outer', 'style-401', 'style-local-116-c33']
Text preview: VERANDA

Ancestor level: 1
Tag: div
Class: ['h-global-transition-all', 'h-heading', 'style-401', 'style-local-116-c33', 'position-relative', 'h-element']
Text preview: VERANDA

Ancestor level: 2
Tag: div
Class: ['w-100', 'h-y-container', 'h-column__content', 'h-column__v-align', 'flex-basis-100', 'align-self-lg-start', 'align-self-md-start', 'align-self-start']
Text preview: Broad and Open VERANDA For a sentimental and harmonious vibe, partake in our wide selection of international and local cuisine. Served à la carte or buffet, have a sumptuous feast while appreciating cultural performances that include music and dancing. Carrying on the theme of traditional Filipino dishes, the Heirloom section of our menu has always been a family favorite. For a more exciting and interactive setting for the guests, enjoy the buffet section with a live cooking station. Veranda is 

In [22]:
outlet_headings = dining_soup.select(
    "div.h-heading__outer.style-401 h2"
)

print("Outlet headings found:", len(outlet_headings))
print()

for outlet_heading in outlet_headings:
    outlet_name = outlet_heading.get_text(
        " ",
        strip=True
    )

    outlet_container = outlet_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    print("OUTLET:", outlet_name)

    for element in outlet_container.find_all(
        ["h4", "p", "a"]
    ):
        element_text = element.get_text(
            " ",
            strip=True
        )

        if element_text:
            print(
                element.name,
                "|",
                element_text,
                "|",
                element.get("href")
            )

    print("=" * 70)

Outlet headings found: 5

OUTLET: VERANDA
h4 | Broad and Open | None
p | For a sentimental and harmonious vibe, partake in our wide selection of international and local cuisine. Served à la carte or buffet, have a sumptuous feast while appreciating cultural performances that include music and dancing. | None
p | Carrying on the theme of traditional Filipino dishes, the Heirloom section of our menu has always been a family favorite. For a more exciting and interactive setting for the guests, enjoy the buffet section with a live cooking station. | None
p | Veranda is a Green Choice Philippines awardee. | None
p | OPERATING HOURS: | None
p | Open daily, 6:00 AM – 10:00 PM | None
p | NOTE: Schedule for buffet may change without prior notice. | None
a | reserve a table | https://forms.office.com/r/LTwLn6BxuU
a | Menu | https://www.taalvistahotel.com/veranda_menu/
OUTLET: TĀZA FRESH TABLE
h4 | Sustainable and Natural | None
p | For a blissful and breezy surrounding, you can find it at TĀZA F

## Initial Dining Record Extraction

The outlet container includes the outlet name, optional tagline, description, operating schedule, optional schedule note, menu link, and optional reservation link.

The raw extraction will preserve the published wording. Schedule inconsistencies and missing values will be documented during data cleaning rather than corrected during collection.

In [23]:
dining_records = []

for outlet_heading in outlet_headings:
    outlet_name = outlet_heading.get_text(
        " ",
        strip=True
    )

    outlet_container = outlet_heading.find_parent(
        "div",
        class_="h-column__content"
    )

    tagline_element = outlet_container.find("h4")

    if tagline_element:
        tagline = tagline_element.get_text(
            " ",
            strip=True
        )
    else:
        tagline = None

    paragraph_texts = []

    for paragraph in outlet_container.find_all("p"):
        paragraph_text = paragraph.get_text(
            " ",
            strip=True
        )

        if paragraph_text:
            paragraph_texts.append(paragraph_text)

    hours_position = None

    for position, paragraph_text in enumerate(
        paragraph_texts
    ):
        if paragraph_text.upper().startswith(
            "OPERATING HOURS"
        ):
            hours_position = position
            break

    if hours_position is not None:
        description_parts = paragraph_texts[
            :hours_position
        ]
        schedule_parts = paragraph_texts[
            hours_position + 1:
        ]
    else:
        description_parts = paragraph_texts
        schedule_parts = []

    schedule_notes = []
    operating_hours_parts = []

    for schedule_text in schedule_parts:
        if schedule_text.upper().startswith("NOTE"):
            schedule_notes.append(schedule_text)
        else:
            operating_hours_parts.append(
                schedule_text
            )

    menu_url = None
    reservation_url = None

    for link in outlet_container.find_all(
        "a",
        href=True
    ):
        link_text = link.get_text(
            " ",
            strip=True
        ).lower()

        if "menu" in link_text:
            menu_url = link["href"]

        if "reserve" in link_text:
            reservation_url = link["href"]

    dining_records.append({
        "outlet_name": outlet_name,
        "tagline": tagline,
        "outlet_description": " ".join(
            description_parts
        ),
        "operating_hours": " | ".join(
            operating_hours_parts
        ),
        "schedule_note": " ".join(
            schedule_notes
        ) or None,
        "menu_url": menu_url,
        "reservation_url": reservation_url,
        "source_url": DINING_URL,
        "date_collected": date.today().isoformat()
    })

dining_raw_df = pd.DataFrame(dining_records)

dining_raw_df.insert(
    0,
    "outlet_record_id",
    range(1, len(dining_raw_df) + 1)
)

print("Rows:", dining_raw_df.shape[0])
print("Columns:", dining_raw_df.shape[1])
print()
print("Missing values:")
print(dining_raw_df.isnull().sum())

display(dining_raw_df)

Rows: 5
Columns: 10

Missing values:
outlet_record_id      0
outlet_name           0
tagline               2
outlet_description    0
operating_hours       0
schedule_note         4
menu_url              0
reservation_url       3
source_url            0
date_collected        0
dtype: int64


,outlet_record_id,outlet_name,tagline,outlet_description,operating_hours,schedule_note,menu_url,reservation_url,source_url,date_collected
0,1,VERANDA,Broad and Open,"For a sentimental and harmonious vibe, partake...","Open daily, 6:00 AM – 10:00 PM",NOTE: Schedule for buffet may change without p...,https://www.taalvistahotel.com/veranda_menu/,https://forms.office.com/r/LTwLn6BxuU,https://www.taalvistahotel.com/dining/,2026-09-02
1,2,TĀZA FRESH TABLE,Sustainable and Natural,"For a blissful and breezy surrounding, you can...","Open daily, 12:00 NN – 10:00 PM",NaN,https://www.taalvistahotel.com/taza_menu,https://forms.office.com/r/LTwLn6BxuU,https://www.taalvistahotel.com/dining/,2026-09-02
2,3,LOBBY LOUNGE,Grand and Iconic,Underneath the remarkable Tudor-designed rooft...,Sunday to Thursday 10:00 AM – 11:00 PM | Frida...,NaN,https://www.taalvistahotel.com/lobby-lounge-menu/,NaN,https://www.taalvistahotel.com/dining/,2026-09-02
3,4,pasalubong,NaN,"For a taste of sweetness, stop by the hotel’s ...",Sunday to Thursday 10:00 AM – 6:00 PM | Friday...,NaN,https://www.taalvistahotel.com/wp-content/uplo...,NaN,https://www.taalvistahotel.com/dining/,2026-09-02
4,5,alta ridge bar,NaN,"Welcome to ALTA Ridge Bar, your perfect spot t...",Friday – Sunday 4:00 PM – 10:00 PM,NaN,https://www.taalvistahotel.com/alta-ridge-bar-...,NaN,https://www.taalvistahotel.com/dining/,2026-09-02


## Raw Dining Data Quality Check

All required dining fields were extracted successfully. Missing optional fields represent information that was not published for every outlet and will not be filled during raw data collection.

The operating schedule for Pasalubong contains overlapping Sunday references. This will be recorded as a source inconsistency during data cleaning.

In [24]:
required_dining_columns = [
    "outlet_name",
    "outlet_description",
    "operating_hours",
    "menu_url",
    "source_url",
    "date_collected"
]

print(
    "Exact duplicate records:",
    dining_raw_df.duplicated().sum()
)

print()
print("Missing required values:")
print(
    dining_raw_df[
        required_dining_columns
    ].isnull().sum()
)

print()
print("Unique outlet names:")
print(
    dining_raw_df["outlet_name"].nunique()
)

Exact duplicate records: 0

Missing required values:
outlet_name           0
outlet_description    0
operating_hours       0
menu_url              0
source_url            0
date_collected        0
dtype: int64

Unique outlet names:
5


In [25]:
dining_file = (
    raw_data_folder
    / "taal_vista_dining_raw.csv"
)

dining_raw_df.to_csv(
    dining_file,
    index=False
)

dining_verification_df = pd.read_csv(
    dining_file
)

print("File saved:", dining_file)
print(
    "Loaded shape:",
    dining_verification_df.shape
)
print("File exists:", dining_file.exists())
print(
    "Duplicate records:",
    dining_verification_df.duplicated().sum()
)

File saved: /Users/jannoelvero/Documents/Taal-Vista-Hotel/data/raw/taal_vista_dining_raw.csv
Loaded shape: (5, 10)
File exists: True
Duplicate records: 0


# Event Venue Data Collection

## Purpose

This section collects publicly available information about Taal Vista Hotel’s event venues.

Potential fields include venue name, dimensions, ceiling height, floor area, and capacities for different event arrangements.

In [26]:
EVENTS_URL = f"{BASE_URL}/events/"

print(
    "Events page allowed:",
    robot_parser.can_fetch(
        HEADERS["User-Agent"],
        EVENTS_URL
    )
)

Events page allowed: True


In [27]:
time.sleep(3)

events_response = requests.get(
    EVENTS_URL,
    headers=HEADERS,
    timeout=30
)

print("Status code:", events_response.status_code)
print("Final URL:", events_response.url)
print("Content type:", events_response.headers.get("Content-Type"))
print("HTML characters:", len(events_response.text))

Status code: 200
Final URL: https://www.taalvistahotel.com/events/
Content type: text/html; charset=UTF-8
HTML characters: 1008067


## Event Table Inspection

The events page contains structured venue tables. Each table will be inspected before selecting or combining data because the page may contain different venue groups, layouts, or repeated responsive versions.

In [28]:
from io import StringIO

event_tables = pd.read_html(
    StringIO(events_response.text)
)

print("HTML tables found:", len(event_tables))

HTML tables found: 4


In [29]:
for table_number, event_table in enumerate(
    event_tables,
    start=1
):
    print()
    print("TABLE:", table_number)
    print("Shape:", event_table.shape)
    print("Columns:")
    print(event_table.columns.tolist())

    display(event_table.head(10))


TABLE: 1
Shape: (17, 10)
Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400
3,BALLROOM 1,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
4,BALLROOM 2,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
5,BALLROOM 3,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
6,COVERED TERRACE,37.90 X 9.60,3.90,363.84,300,NaN,NaN,NaN,220,450
7,CAMIA,8.50 X 6.80,3.40,57.80,50,32,22,22,30,60
8,LILY,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60
9,SANTAN,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60



TABLE: 2
Shape: (14, 10)
Columns:
['VENUE', 'DIMENSION (IN METERS)', 'FLOOR AREA (IN SQ. METERS)', 'CONFERENCE CAPACITY', 'THEATER', 'CLASSROOM', 'U-SHAPE', 'BLOCK', 'BANQUET', 'COCKTAIL']


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,360,180.0,90.0,90.0,270,270
1,BALLROOM 1,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
2,BALLROOM 2,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
3,BALLROOM 3,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
4,COVERED TERRACE,37.90 X 9.60,3.90,363.84,100,NaN,NaN,NaN,80,80
5,CAMIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
6,LILY,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
7,SANTAN,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
8,DAHLIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
9,CHAMPACA,7.70 X 6.55,3.25,50.44,20,14.0,10.0,12.0,10,10



TABLE: 3
Shape: (11, 10)
Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.40,420,320,100,120,250,550
3,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.38,NaN,NaN,NaN,NaN,NaN,70
4,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.9960,100,NaN,NaN,NaN,60,120
5,WALING-WALING 1,7.60 X 7.10,3.25,53.96,42,30,20,15,30,60
6,WALING-WALING 2,7.40 X 7.10,3.25,52.54,42,30,20,15,30,60
7,ROSAL 1,9.65 X 7.95,3.20,76.72,60,30,20,15,30,80
8,ROSAL 2,7.60 X 7.95,3.20,60.42,50,30,20,15,30,70
9,GUMAMELA,8.40 X 5.00,3.20,42.00,NaN,NaN,NaN,12,NaN,NaN



TABLE: 4
Shape: (9, 10)
Columns:
['VENUE', 'DIMENSION (IN METERS)', 'FLOOR AREA (IN SQ. METERS)', 'CONFERENCE CAPACITY', 'THEATER', 'CLASSROOM', 'U-SHAPE', 'BLOCK', 'BANQUET', 'COCKTAIL']


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.400,120.0,80.0,30.0,30.0,100.0,100.0
1,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.380,NaN,NaN,NaN,NaN,NaN,30.0
2,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.996,60.0,NaN,NaN,NaN,30.0,30.0
3,WALING-WALING 1,7.60 X 7.10,3.25,53.960,24.0,16.0,13.0,14.0,15.0,15.0
4,WALING-WALING 2,7.40 X 7.10,3.25,52.540,24.0,16.0,13.0,14.0,15.0,15.0
5,ROSAL 1,9.65 X 7.95,3.20,76.720,24.0,16.0,13.0,14.0,15.0,15.0
6,ROSAL 2,7.60 X 7.95,3.20,60.420,24.0,16.0,13.0,14.0,15.0,15.0
7,GUMAMELA,8.40 X 5.00,3.20,42.000,NaN,NaN,NaN,7.0,NaN,NaN
8,JASMIN,7.70 X 5.15,3.50,39.660,16.0,12.0,10.0,10.0,10.0,10.0


In [30]:
display(event_tables[0])
display(event_tables[1])
display(event_tables[2])

,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,1000,750,200,260,800,1400
3,BALLROOM 1,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
4,BALLROOM 2,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
5,BALLROOM 3,29.50 X 14.60,5.50,430.70,350,220,80,100,220,450
6,COVERED TERRACE,37.90 X 9.60,3.90,363.84,300,NaN,NaN,NaN,220,450
7,CAMIA,8.50 X 6.80,3.40,57.80,50,32,22,22,30,60
8,LILY,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60
9,SANTAN,8.30 X 7.00,3.40,58.10,50,32,22,22,30,60


,VENUE,DIMENSION (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
0,GRAND BALLROOM,43.80 X 29.50,5.50,1292.10,360,180.0,90.0,90.0,270,270
1,BALLROOM 1,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
2,BALLROOM 2,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
3,BALLROOM 3,29.50 X 14.60,5.50,430.70,120,60.0,30.0,30.0,90,90
4,COVERED TERRACE,37.90 X 9.60,3.90,363.84,100,NaN,NaN,NaN,80,80
5,CAMIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
6,LILY,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
7,SANTAN,8.30 X 7.00,3.40,58.10,24,12.0,10.0,10.0,15,15
8,DAHLIA,8.50 X 6.80,3.40,57.80,24,12.0,10.0,10.0,15,15
9,CHAMPACA,7.70 X 6.55,3.25,50.44,20,14.0,10.0,12.0,10,10


,0,1,2,3,4,5,6,7,8,9
0,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,CONFERENCE CAPACITY,BANQUET,COCKTAIL
1,VENUE,DIMENSION (IN METERS),CEILING HEIGHT (IN METERS),FLOOR AREA (IN SQ. METERS),THEATER,CLASSROOM,U-SHAPE,BLOCK,BANQUET,COCKTAIL
2,SAMPAGUITA BALLROOM,30.77 X 18.70,3.90,575.40,420,320,100,120,250,550
3,SAMPAGUITA FOYER,15.30 X 4.60,3.88,70.38,NaN,NaN,NaN,NaN,NaN,70
4,LOWER LEVEL FOYER,22.20 X 5.45,3.40,120.9960,100,NaN,NaN,NaN,60,120
5,WALING-WALING 1,7.60 X 7.10,3.25,53.96,42,30,20,15,30,60
6,WALING-WALING 2,7.40 X 7.10,3.25,52.54,42,30,20,15,30,60
7,ROSAL 1,9.65 X 7.95,3.20,76.72,60,30,20,15,30,80
8,ROSAL 2,7.60 X 7.95,3.20,60.42,50,30,20,15,30,70
9,GUMAMELA,8.40 X 5.00,3.20,42.00,NaN,NaN,NaN,12,NaN,NaN


## Event Table Context Issue

The events page contains two venue groups and two different capacity sets for each group.

The capacity values will not be combined until the surrounding page labels are identified. The differences may represent event categories, layouts, or another operational distinction.

In [31]:
events_soup = BeautifulSoup(
    events_response.text,
    "lxml"
)

html_tables = events_soup.find_all("table")

print("BeautifulSoup tables found:", len(html_tables))
print()

for table_number, html_table in enumerate(
    html_tables,
    start=1
):
    print("TABLE:", table_number)
    print("Table class:", html_table.get("class"))
    print("Table id:", html_table.get("id"))
    print()

    previous_headings = html_table.find_all_previous(
        ["h1", "h2", "h3", "h4", "h5"],
        limit=8
    )

    print("Nearest previous headings:")

    for previous_heading in reversed(
        previous_headings
    ):
        heading_text = previous_heading.get_text(
            " ",
            strip=True
        )

        if heading_text:
            print(
                previous_heading.name,
                "|",
                heading_text
            )

    print("=" * 70)

BeautifulSoup tables found: 4

TABLE: 1
Table class: ['tablepress', 'tablepress-id-1', 'tbody-has-connected-cells']
Table id: tablepress-1

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 2
Table class: ['tablepress', 'tablepress-id-3']
Table id: tablepress-3

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 3
Table class: ['tablepress', 'tablepress-id-4', 'tbody-has-connected-cells']
Table id: tablepress-4

Nearest previous headings:
h3 | blue lace agate
h3 | Lemon quartz
h3 | green jade
h3 | CORPORATE MEETINGS
h3 | A HAPPY BIRTHDAY
h3 | PROM NIGHT PARTY
h3 | FOR THE DEBUTANTE
h3 | Events Space
TABLE: 4
Table class: ['tablepress', 'tablepress-id-5']
Table id: tablepress-5

Nearest previous h

In [32]:
for table_number, html_table in enumerate(
    html_tables,
    start=1
):
    print("TABLE:", table_number)

    current_element = html_table

    for level in range(1, 7):
        current_element = current_element.parent

        if current_element is None:
            break

        print()
        print("Parent level:", level)
        print("Tag:", current_element.name)
        print("ID:", current_element.get("id"))
        print("Class:", current_element.get("class"))

        previous_sibling = (
            current_element.find_previous_sibling()
        )

        if previous_sibling:
            sibling_text = previous_sibling.get_text(
                " ",
                strip=True
            )

            print(
                "Previous sibling text:",
                sibling_text[:300]
            )
        else:
            print("Previous sibling text: None")

    print()
    print("=" * 70)

TABLE: 1

Parent level: 1
Tag: div
ID: None
Class: []
Previous sibling text: None

Parent level: 2
Tag: div
ID: None
Class: ['style-1996', 'style-local-135-c106', 'h-hide-md', 'h-hide-sm', 'position-relative', 'h-element']
Previous sibling text: 

Parent level: 3
Tag: div
ID: None
Class: ['w-100', 'h-y-container']
Previous sibling text: None

Parent level: 4
Tag: div
ID: lake-wing
Class: ['h-tabs-item-content', 'h-tabs-item', 'h-tabs-content-horizontal', 'h-tabs-content-135-c104', 'style-1617', 'style-local-135-c104', 'position-relative', 'h-element', 'h-tabs-content-active']
Previous sibling text: LAKE WING MOUNTAIN WING

Parent level: 5
Tag: div
ID: None
Class: ['h-tabs', 'h-tabs-horizontal', 'h-tabs--horizontal--stretch-lg', 'h-tabs--horizontal--stretch-md', 'h-tabs--horizontal--full', 'h-tabs', 'h-tabs-horizontal', 'h-tabs--horizontal--stretch-lg', 'h-tabs--horizontal--stretch-md', 'h-tabs--horizontal--full', 'style-1616', 'style-local-135-c103', 'position-relative', 'h-element']
P

## Event Capacity Data Inconsistency

The events page uses separate tabs for Lake Wing and Mountain Wing. Each wing contains a desktop table and a mobile table.

The desktop and mobile versions contain different capacity values for the same venues. Both versions will be preserved as separate raw datasets.

No capacity value will be corrected or selected during data collection. The differences will be quantified and documented during data cleaning and business analysis.

In [33]:
event_table_details = [
    {
        "table_index": 0,
        "hotel_wing": "Lake Wing",
        "display_version": "Desktop",
        "file_name": "taal_vista_events_lake_desktop_raw.csv"
    },
    {
        "table_index": 1,
        "hotel_wing": "Lake Wing",
        "display_version": "Mobile",
        "file_name": "taal_vista_events_lake_mobile_raw.csv"
    },
    {
        "table_index": 2,
        "hotel_wing": "Mountain Wing",
        "display_version": "Desktop",
        "file_name": "taal_vista_events_mountain_desktop_raw.csv"
    },
    {
        "table_index": 3,
        "hotel_wing": "Mountain Wing",
        "display_version": "Mobile",
        "file_name": "taal_vista_events_mountain_mobile_raw.csv"
    }
]

saved_event_files = []

for table_detail in event_table_details:
    event_raw_df = event_tables[
        table_detail["table_index"]
    ].copy()

    event_raw_df["hotel_wing"] = (
        table_detail["hotel_wing"]
    )

    event_raw_df["display_version"] = (
        table_detail["display_version"]
    )

    event_raw_df["source_url"] = EVENTS_URL
    event_raw_df["date_collected"] = (
        date.today().isoformat()
    )

    event_file = (
        raw_data_folder
        / table_detail["file_name"]
    )

    event_raw_df.to_csv(
        event_file,
        index=False
    )

    saved_event_files.append(event_file)

    print("File:", event_file.name)
    print("Shape:", event_raw_df.shape)
    print("Exists:", event_file.exists())
    print()

File: taal_vista_events_lake_desktop_raw.csv
Shape: (17, 14)
Exists: True

File: taal_vista_events_lake_mobile_raw.csv
Shape: (14, 14)
Exists: True

File: taal_vista_events_mountain_desktop_raw.csv
Shape: (11, 14)
Exists: True

File: taal_vista_events_mountain_mobile_raw.csv
Shape: (9, 14)
Exists: True

